# Can we predict late delivery?
Protocol: `protocol.yaml` (temporal holdout, 5-split walk-forward CV on training rows, choose by mean CV ROC AUC).

`claim_paid_eur` is excluded: a claim settled for the consignment is only known after delivery, so it is not available at prediction time (and leaks the outcome). `consignment` (ID) and `dispatched_on` (split only) are excluded per protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd, yaml
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

proto = yaml.safe_load(open('protocol.yaml'))
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest() == proto['dataset']['sha256']
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest() == proto['split']['holdout_sha256']
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df) == proto['dataset']['rows'] and len(hold_idx) == proto['split']['holdout_rows']
df['t'] = pd.to_datetime(df['dispatched_on'])
is_hold = np.zeros(len(df), bool); is_hold[hold_idx] = True
train = df[~is_hold].sort_values('t', kind='stable')   # index = original 0-based row positions
hold = df[is_hold]
print(len(train), len(hold), train.t.max(), hold.t.min(), train.late_delivery.mean())

In [ ]:
FEATURES = ['carrier', 'distance_km', 'weight_kg', 'parcels']   # known at dispatch
num = ['distance_km', 'weight_kg', 'parcels']; cat = ['carrier']
X, y = train[FEATURES], train['late_delivery'].to_numpy()
Xh, yh = hold[FEATURES], hold['late_delivery'].to_numpy()

## Candidates (named before any scoring)
All preprocessing lives inside the pipelines, so it is fitted within each CV fold.

In [ ]:
def prep(scale):
    return ColumnTransformer([('n', StandardScaler() if scale else 'passthrough', num),
                              ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
candidates = {
    'baseline_prior': DummyClassifier(strategy='prior'),
    'logreg_C0.1': make_pipeline(prep(True), LogisticRegression(C=0.1, max_iter=1000)),
    'logreg_C1': make_pipeline(prep(True), LogisticRegression(C=1.0, max_iter=1000)),
    'rf_depth6': make_pipeline(prep(False), RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=0, n_jobs=1)),
    'hgb_small': make_pipeline(prep(False), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=0)),
}
print(list(candidates))

In [ ]:
tscv = TimeSeriesSplit(n_splits=proto['cv']['n_splits'])
folds = list(tscv.split(X))
cv_scores = {}
for name, m in candidates.items():
    s = []
    for tr, va in folds:
        mm = clone(m).fit(X.iloc[tr], y[tr])
        s.append(float(roc_auc_score(y[va], mm.predict_proba(X.iloc[va])[:, 1])))
    cv_scores[name] = s
cv = pd.DataFrame(cv_scores).T; cv['mean'] = cv.mean(axis=1); cv

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen, cv.loc[chosen, 'mean'])
final = clone(candidates[chosen]).fit(X, y)
holdout_auc = float(roc_auc_score(yh, final.predict_proba(Xh)[:, 1]))   # scored once
print('holdout roc_auc:', holdout_auc)

In [ ]:
evidence = {
    'features': FEATURES,
    'candidates': list(candidates),
    'cv_folds': [[[int(i) for i in train.index[tr]], [int(i) for i in train.index[va]]] for tr, va in folds],
    'cv_scores': cv_scores,
    'chosen': chosen,
    'holdout_rows': [int(i) for i in hold.index],
    'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json', 'w'))

## Conclusion
See the CV table and holdout ROC AUC above: the headline is the holdout ROC AUC of the CV-chosen candidate, refitted on all training rows.